# CS781 Phase 3 - Rhetorical Roles - notebook `D`

**Full design:** sentences (Phase-1 tokenization, <=128 tokens) packed into 512-token windows -> Phase-1 backbone ->
mean-pooled sentence vectors + relative position (16 sinusoids) + CL/IT embedding -> BiLSTM over a range of ~70-100
consecutive sentences (3,072 tokens) -> 13 labels; cross-entropy with sqrt class weights (cap 10).
**Base technique:** LoRA r16 (lr 5e-4). Experiments 15-25 (design ablations, seeds) each change one thing from the base.
Training on CL_train + IT_train only; the epoch (max 10, patience 4) is chosen on CL_dev + IT_dev with the official metric
(macro-F1 over all 13 labels). Dev and test are never trained on.

| ID | GPU | run | technique | change vs full design | question |
|---|---|---|---|---|---|
| D03 | GPU0 | `D03_lorma_lr2e4` | LoRMA r16 | lr 2e-4 | LoRMA collapsed on A and B at lr 5e-4 in the first runs |
| D05 | GPU0 | `D05_rslora64` | rsLoRA r64 | - | rank-stabilised scaling at a large rank |
| D09 | GPU0 | `D09_prompt` | Prompt tuning (20) | - | input-only adaptation, frozen backbone |
| D11 | GPU0 | `D11_adapter` | Houlsby adapter | - | bottleneck adapters |
| D12 | GPU0 | `D12_bitfit` | BitFit (norm gains) | - | sparse tuning: only the normalisation gains |
| D14 | GPU0 | `D14_full` | Full FT | - | reference: every weight trains |
| D15 | GPU0 | `D15_lora_abl_sentonly` | LoRA r16 | sentence alone, MLP (range 1024) | no context at all |
| D01 | GPU1 | `D01_lora` | LoRA r16 | - | base: the only method that trained stably on all five backbones in the first runs |
| D02 | GPU1 | `D02_lorma` | LoRMA r16 | - | compulsory technique: multiplicative low-rank update |
| D04 | GPU1 | `D04_lorma32` | LoRMA r32 | rank 32 | rank sensitivity of LoRMA |
| D06 | GPU1 | `D06_adalora` | AdaLoRA r12->8 | - | adaptive rank budget across matrices |
| D07 | GPU1 | `D07_qlora` | QLoRA (NF4 + LoRA r16) | - | 4-bit frozen backbone: cost of quantisation |
| D08 | GPU1 | `D08_ia3` | IA3 | - | ~0.05M-parameter rescaling vs low-rank updates |
| D10 | GPU1 | `D10_prefix` | Prefix tuning (20) | - | learned prefix states at every layer vs input only |
| D13 | GPU1 | `D13_partial` | Partial FT (top 4) | - | layer-wise tuning: top 4 blocks only |

**Inputs:** dataset `cs781-p3-code` (latest version), `cs781-legal-data`, Phase-1 checkpoint of model D.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~2.7 h after a 3-5 min smoke test. A run that fails the smoke test is dropped (its error is in the
report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt` (trained parameters + config + best epoch +
dev score), `bundle_D.zip` (dev/test probabilities + reports, for the ensemble), `REPORT_D.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS, MODELS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs", f"{W}/models"
CACHE = f"{W}/rr_cache.pkl"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions"):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-3 data from Hugging Face
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
print("p3 data:", sorted(os.listdir(P3DATA)))

# dev truth files in the official evaluator's format (used to cross-check our metric)
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- tokenize every split once (shared by all jobs) + CPU tests
t0 = time.time()
cache = rr_data.build_cache(P3DATA, TOK_ROOT, CACHE)
for k, v in cache["docs"].items():
    print(k, len(v), "docs", sum(len(x["sents"]) for x in v), "sentences")
print(f"cache built in {time.time() - t0:.0f}s")
r = subprocess.run("python tests_rr.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'evaluate_phase3.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'rr_data.py', 'rr_ensemble.py', 'rr_eval.py', 'rr_model.py', 'rr_train.py', 'tests_rr.py']


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

p3 data: ['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

('CL', 'train') 40 docs 10593 sentences
('CL', 'dev') 5 docs 1396 sentences
('CL', 'test') 5 docs 1339 sentences
('IT', 'train') 40 docs 6252 sentences
('IT', 'dev') 5 docs 746 sentences
('IT', 'test') 5 docs 858 sentences
cache built in 3s
ok test_segments
ok test_pack_and_arrays
ok test_metric
ok test_files_and_validation
 


In [2]:
def cmd(model, out, extra, model_dir):
    return (f"python rr_train.py --ckpt {CKPT[model]} --tok_root {TOK_ROOT} --data_dir {P3DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: 1 tiny epoch per experiment."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            extra = e["args"] + (" --max_docs 1 --max_sents 40 --epochs 1" if smoke else "")
            if smoke and e is not jobs[0]:
                extra += " --no_test"
            status[t] = run(t, cmd(e["model"], f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Runs 1 tiny epoch of every experiment. A failing experiment is dropped (its error is printed and
    reported) instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = 'D'
QUEUES = [[{'id': 'D03',
   'tag': 'D03_lorma_lr2e4',
   'model': 'd',
   'tech': 'LoRMA r16',
   'change': 'lr 2e-4',
   'args': '--method lorma --rank 16 --alpha 32 --lr 2e-4'},
  {'id': 'D05', 'tag': 'D05_rslora64', 'model': 'd', 'tech': 'rsLoRA r64', 'change': '-', 'args': '--method rslora --rank 64 --alpha 16'},
  {'id': 'D09', 'tag': 'D09_prompt', 'model': 'd', 'tech': 'Prompt tuning (20)', 'change': '-', 'args': '--method prompt --n_prompt 20'},
  {'id': 'D11', 'tag': 'D11_adapter', 'model': 'd', 'tech': 'Houlsby adapter', 'change': '-', 'args': '--method adapter --bottleneck 64'},
  {'id': 'D12', 'tag': 'D12_bitfit', 'model': 'd', 'tech': 'BitFit (norm gains)', 'change': '-', 'args': '--method norm'},
  {'id': 'D14', 'tag': 'D14_full', 'model': 'd', 'tech': 'Full FT', 'change': '-', 'args': '--method full'},
  {'id': 'D15',
   'tag': 'D15_lora_abl_sentonly',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'sentence alone, MLP (range 1024)',
   'args': '--method lora --rank 16 --alpha 32 --mode single --head mlp --seg_tokens 1024'}],
 [{'id': 'D01', 'tag': 'D01_lora', 'model': 'd', 'tech': 'LoRA r16', 'change': '-', 'args': '--method lora --rank 16 --alpha 32'},
  {'id': 'D02', 'tag': 'D02_lorma', 'model': 'd', 'tech': 'LoRMA r16', 'change': '-', 'args': '--method lorma --rank 16 --alpha 32'},
  {'id': 'D04', 'tag': 'D04_lorma32', 'model': 'd', 'tech': 'LoRMA r32', 'change': 'rank 32', 'args': '--method lorma --rank 32 --alpha 64'},
  {'id': 'D06',
   'tag': 'D06_adalora',
   'model': 'd',
   'tech': 'AdaLoRA r12->8',
   'change': '-',
   'args': '--method adalora --rank 12 --alpha 24 --target_rank 8'},
  {'id': 'D07', 'tag': 'D07_qlora', 'model': 'd', 'tech': 'QLoRA (NF4 + LoRA r16)', 'change': '-', 'args': '--method qlora --rank 16 --alpha 32'},
  {'id': 'D08', 'tag': 'D08_ia3', 'model': 'd', 'tech': 'IA3', 'change': '-', 'args': '--method ia3'},
  {'id': 'D10', 'tag': 'D10_prefix', 'model': 'd', 'tech': 'Prefix tuning (20)', 'change': '-', 'args': '--method prefix --n_prompt 20'},
  {'id': 'D13', 'tag': 'D13_partial', 'model': 'd', 'tech': 'Partial FT (top 4)', 'change': '-', 'args': '--method partial --top_k 4'}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU0] smoke_D03_lorma_lr2e4: exit 0 | ion.zip  {'Fact': 2176, 'Issue': 0, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 1, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 20, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [05:58:52] DONE
[GPU1] smoke_D01_lora: exit 0 | ion.zip  {'Fact': 2135, 'Issue': 9, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 53, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [05:58:53] DONE
[GPU1] smoke_D02_lorma: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:16 RulingByLowerCourt->Fact:4 None->Fact:3 Issue->Fact:3 Statute->Fact:1 / [05:59:05] DONE
[GPU0] smoke_D05_rslora64: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:16 RulingByLowerCourt->Fact:4 None->Fac

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU1] D01_lora: exit 0 | act': 376, 'Issue': 11, 'ArgumentPetitioner': 251, 'ArgumentRespondent': 81, 'Statute': 147, 'Dissent': 4, 'PrecedentReliedUpon': 412, 'PrecedentNotReliedUpon': 18, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 70, 'RatioOfTheDecision': 753, 'RulingByPresentCourt': 45, 'None': 29} / [06:19:01] DONE
[GPU0] D03_lorma_lr2e4: exit 0 | act': 439, 'Issue': 12, 'ArgumentPetitioner': 242, 'ArgumentRespondent': 74, 'Statute': 160, 'Dissent': 0, 'PrecedentReliedUpon': 355, 'PrecedentNotReliedUpon': 27, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 86, 'RatioOfTheDecision': 741, 'RulingByPresentCourt': 41, 'None': 20} / [06:20:46] DONE
[GPU1] D02_lorma: exit 0 | t': 401, 'Issue': 16, 'ArgumentPetitioner': 163, 'ArgumentRespondent': 154, 'Statute': 198, 'Dissent': 0, 'PrecedentReliedUpon': 357, 'PrecedentNotReliedUpon': 24, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 119, 'RatioOfTheDecision': 707, 'RulingByPresentCourt': 31, 'None': 27} / [06:37:46] DONE
[GPU0] D05_

## 3. Report

In [6]:
SHORT = {"ArgumentPetitioner": "ArgPet", "ArgumentRespondent": "ArgResp", "PrecedentReliedUpon": "PrecRel",
         "PrecedentNotReliedUpon": "PrecNRel", "PrecedentOverruled": "PrecOvr", "RulingByLowerCourt": "RulLower",
         "RatioOfTheDecision": "Ratio", "RulingByPresentCourt": "RulPres"}

def official(tag):
    """Official evaluator on this run's dev predictions (cross-check of our metric)."""
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {RUNS}/{tag}/dev_pred.json --labels {P3DATA}/label_vocab.json",
                       shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem"]
    rows = []
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SMOKE FAILED (dropped)" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:28s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]
        off = official(e["tag"])
        rows.append((e, s))
        L.append(f"{e['id']} | {e['tag']:28s} | {e['tech'][:18]:18s} | {e['change'][:26]:26s} | {dv['macro_f1']:.4f} | {off:.4f}   | "
                 f"{dv['macro_f1_CL']:.4f} | {dv['macro_f1_IT']:.4f} | {dv['macro_p']:.4f} | {dv['macro_r']:.4f} | {s['best_epoch']:2d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | {s.get('peak_mem_gb') or 0:.1f}G")
        shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    if rows:
        labs = rows[0][1]["labels"]
        L.append("\nper-label dev F1 (official: all 13 labels count; Dissent and PrecedentOverruled have no dev sentences -> 0)")
        L.append(f"{'ID':4s} " + " ".join(f"{SHORT.get(l, l)[:8]:>8s}" for l in labs))
        for e, s in rows:
            pl = s["dev"]["per_label"]
            L.append(f"{e['id']:4s} " + " ".join(f"{pl[l]['f1']:8.2f}" for l in labs))
        L.append("\ntop confusions (gold->pred:count)")
        for e, s in rows:
            L.append(f"{e['id']}: " + " ".join(s["dev"]["top_confusions"][:6]))
    bundle = f"{W}/bundle_{name}.zip"                     # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT D ===
ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem
D03 | D03_lorma_lr2e4              | LoRMA r16          | lr 2e-4                    | 0.5288 | 0.5288   | 0.4778 | 0.5190 | 0.5637 | 0.5149 |  6 |     5.51M |   20 |  5,612 | 5.4G
D05 | D05_rslora64                 | rsLoRA r64         | -                          | 0.4994 | 0.4994   | 0.3876 | 0.5212 | 0.5569 | 0.4845 |  6 |    13.46M |   21 |  5,545 | 5.2G
D09 | D09_prompt                   | Prompt tuning (20) | -                          | 0.4725 | 0.4725   | 0.3452 | 0.4924 | 0.5282 | 0.4479 |  7 |     2.19M |   17 |  6,984 | 4.9G
D11 | D11_adapter                  | Houlsby adapter    | -                          | 0.5039 | 0.5039   | 0.3843 | 0.5185 | 0.5384 | 0.4975 |  8 |     4.55M |   17 |  6,870 | 4.2G
D12 | D12_bitfit                   | BitFit (norm gains | -                    